# Phase 3: YOLOv8 Layout Detection Training - Google Colab

**Project**: Image Preprocessing Detector  
**Phase**: 3 - ML for Document Layout Detection  
**Model**: YOLOv8n/YOLOv8s  
**Task**: Object detection (4 classes: table, image, handwriting, formula)  

## Overview

This notebook trains a YOLOv8 object detector to identify document elements:
- Tables
- Images/Figures
- Handwriting regions
- Mathematical Formulas

## Colab Pro Requirements

- **Session Limit**: 12 hours
- **GPU**: V100 (16GB) or T4 (15GB)
- **Expected Training Time**: 50-80 hours (4-7 sessions with resume)
- **Google Drive Space**: ~50GB (dataset + checkpoints)

## Important: YOLOv8 Training is Multi-Session

YOLOv8 typically requires 100+ epochs. With 12-hour sessions:
- **V100**: ~15-20 epochs per session → 5-7 sessions needed
- **T4**: ~10-15 epochs per session → 7-10 sessions needed

The notebook automatically resumes from the last checkpoint.

---

## Cell 1: Environment Setup & GPU Check

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Cell 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

## Cell 3: Install Ultralytics YOLOv8

In [ ]:
!pip install -q ultralytics>=8.0.0
!pip install -q tensorboard

# Clone project repository (if needed)
# !git clone https://github.com/YOUR_USERNAME/image-preprocessing-detector.git
# %cd /content/image-preprocessing-detector

from ultralytics import YOLO
print("✅ Ultralytics YOLOv8 installed!")

## Cell 4: Setup Paths & Configuration

In [ ]:
import yaml
from pathlib import Path

# Google Drive paths
DRIVE_ROOT = "/content/drive/MyDrive/image-preprocessing-detector"
DATASET_PATH = f"{DRIVE_ROOT}/datasets/layout_phase3"
DATASET_YAML = f"{DATASET_PATH}/dataset.yaml"
CHECKPOINT_DIR = f"{DRIVE_ROOT}/checkpoints/phase3_yolov8"
MODEL_OUTPUT_DIR = f"{DRIVE_ROOT}/models/phase3_yolov8"

# Create directories
for path in [CHECKPOINT_DIR, MODEL_OUTPUT_DIR]:
    Path(path).mkdir(parents=True, exist_ok=True)

# Load config
config_path = "/content/drive/MyDrive/image-preprocessing-detector/configs/colab_phase3_yolov8.yaml"
if Path(config_path).exists():
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    print(f"✅ Loaded config: {config['model']['variant']}")
    MODEL_VARIANT = config['model']['variant']
    BATCH_SIZE = config['training']['batch_size']
    EPOCHS = config['training']['epochs']
else:
    print("⚠️  Config not found, using defaults")
    MODEL_VARIANT = "yolov8n"
    BATCH_SIZE = 16
    EPOCHS = 100

print(f"\n📁 Paths:")
print(f"   Dataset: {DATASET_PATH}")
print(f"   Checkpoints: {CHECKPOINT_DIR}")
print(f"   Models: {MODEL_OUTPUT_DIR}")
print(f"\n⚙️  Config:")
print(f"   Model: {MODEL_VARIANT}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Epochs: {EPOCHS}")

## Cell 5: Verify Dataset

**Note**: Dataset must be in YOLO format with `dataset.yaml`

In [ ]:
# Check if dataset exists
if not Path(DATASET_YAML).exists():
    print(f"❌ Dataset YAML not found: {DATASET_YAML}")
    print("\nPlease prepare your dataset in YOLO format:")
    print("  dataset_root/")
    print("    ├── dataset.yaml")
    print("    ├── train/")
    print("    │   ├── images/")
    print("    │   └── labels/")
    print("    └── val/")
    print("        ├── images/")
        print("        └── labels/")
    raise FileNotFoundError("Dataset not found")

# Load and verify dataset config
with open(DATASET_YAML, 'r') as f:
    dataset_config = yaml.safe_load(f)

print("✅ Dataset found!")
print(f"\n📊 Dataset Info:")
print(f"   Classes: {dataset_config.get('names', [])}")
print(f"   Num classes: {dataset_config.get('nc', 0)}")
print(f"   Train path: {dataset_config.get('train', 'N/A')}")
print(f"   Val path: {dataset_config.get('val', 'N/A')}")

# List some sample images
train_images = Path(DATASET_PATH) / "train" / "images"
if train_images.exists():
    sample_images = list(train_images.glob("*.jpg"))[:5]
    print(f"\n📷 Sample images: {len(sample_images)} shown")
    for img in sample_images:
        print(f"     {img.name}")

## Cell 6: Initialize YOLOv8 Model

In [ ]:
from ultralytics import YOLO

# Check for existing checkpoint to resume
last_checkpoint = Path(CHECKPOINT_DIR) / "last.pt"

if last_checkpoint.exists():
    print(f"\n📂 Found checkpoint! Resuming training from: {last_checkpoint}")
    model = YOLO(str(last_checkpoint))
    RESUME = True
else:
    print(f"\n🆕 No checkpoint found. Starting from pretrained {MODEL_VARIANT}")
    model = YOLO(f"{MODEL_VARIANT}.pt")  # Downloads pretrained weights
    RESUME = False

print("\n📊 Model Info:")
print(f"   Variant: {MODEL_VARIANT}")
print(f"   Resume: {RESUME}")
model.info()  # Print model summary

## Cell 7: Start Training

**This cell will run for ~11.5 hours then auto-save.**  
**Simply re-run this cell in a new session to continue training.**

In [ ]:
import time
import signal
from datetime import datetime

# Calculate max training time (11.5 hours = 41,400 seconds)
MAX_TRAINING_SECONDS = 11.5 * 3600
start_time = time.time()

print("\n" + "="*60)
print("🚀 STARTING YOLOV8 TRAINING")
print("="*60)
print(f"\nStart time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Max duration: 11.5 hours")
print(f"Model: {MODEL_VARIANT}")
print(f"Resume: {RESUME}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print("\n⚠️  Training will auto-stop at 11.5 hours to save checkpoint.")
print("   Simply re-run this cell in a new session to continue.\n")

# Train the model
try:
    results = model.train(
        data=DATASET_YAML,
        epochs=EPOCHS,
        imgsz=640,
        batch=BATCH_SIZE,
        name='phase3_yolov8',
        project=CHECKPOINT_DIR,
        resume=RESUME,
        patience=50,  # Early stopping patience
        save=True,
        save_period=10,  # Save checkpoint every 10 epochs
        device=0,  # Use first GPU
        workers=2,  # Colab CPU cores limited
        optimizer='SGD',
        lr0=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,
        cos_lr=True,  # Cosine LR scheduler
        amp=True,  # Mixed precision
        # Augmentation settings
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        fliplr=0.5,
        mosaic=1.0,
        # Time limit callback
        time_limit=MAX_TRAINING_SECONDS / 3600,  # hours
    )
    
    print("\n" + "="*60)
    print("✅ TRAINING COMPLETED (or time limit reached)")
    print("="*60)
    
except KeyboardInterrupt:
    print("\n⚠️  Training interrupted by user")
except Exception as e:
    print(f"\n❌ Training error: {e}")

# Print final status
elapsed_hours = (time.time() - start_time) / 3600
print(f"\n⏱️  Training duration: {elapsed_hours:.2f} hours")
print(f"   Checkpoint saved to: {CHECKPOINT_DIR}")
print(f"\n🔄 To continue training:")
print("   1. Start a new Colab session")
print("   2. Re-run all cells (will auto-resume from last.pt)")

## Cell 8: Validate Model

In [ ]:
# Load best model
best_model_path = Path(CHECKPOINT_DIR) / "phase3_yolov8" / "weights" / "best.pt"

if best_model_path.exists():
    print(f"\n📊 Validating best model: {best_model_path}")
    model = YOLO(str(best_model_path))
    
    # Run validation
    metrics = model.val(data=DATASET_YAML)
    
    print("\n📈 Validation Metrics:")
    print(f"   mAP50: {metrics.box.map50:.4f}")
    print(f"   mAP50-95: {metrics.box.map:.4f}")
    print(f"   Precision: {metrics.box.mp:.4f}")
    print(f"   Recall: {metrics.box.mr:.4f}")
else:
    print("⚠️  Best model not found yet. Continue training.")

## Cell 9: Export to ONNX

In [ ]:
# Export best model to ONNX
if best_model_path.exists():
    print("\n🔄 Exporting model to ONNX...")
    
    model = YOLO(str(best_model_path))
    onnx_path = model.export(
        format='onnx',
        imgsz=640,
        opset=12,
        simplify=True,
        dynamic=True
    )
    
    print(f"✅ ONNX model exported: {onnx_path}")
    
    # Copy to output directory
    import shutil
    final_path = Path(MODEL_OUTPUT_DIR) / f"{MODEL_VARIANT}_best.onnx"
    shutil.copy(onnx_path, final_path)
    print(f"✅ Copied to: {final_path}")
    
    # Check file size
    size_mb = final_path.stat().st_size / (1024 * 1024)
    print(f"   Model size: {size_mb:.2f} MB")
else:
    print("⚠️  Complete training first before exporting")

## Cell 10: Training Summary

In [ ]:
print("\n" + "="*60)
print("🎉 PHASE 3 YOLOV8 TRAINING SESSION COMPLETE")
print("="*60)

print("\n📁 Output Locations:")
print(f"   Checkpoints: {CHECKPOINT_DIR}/phase3_yolov8/")
print(f"   Best weights: {CHECKPOINT_DIR}/phase3_yolov8/weights/best.pt")
print(f"   ONNX model: {MODEL_OUTPUT_DIR}/{MODEL_VARIANT}_best.onnx")

# Check training progress
last_checkpoint = Path(CHECKPOINT_DIR) / "last.pt"
if last_checkpoint.exists():
    checkpoint = torch.load(last_checkpoint)
    current_epoch = checkpoint.get('epoch', 0)
    print(f"\n📊 Progress:")
    print(f"   Current epoch: {current_epoch}/{EPOCHS}")
    print(f"   Completion: {(current_epoch/EPOCHS)*100:.1f}%")
    
    if current_epoch < EPOCHS:
        remaining = EPOCHS - current_epoch
        print(f"\n🔄 Training Not Complete:")
        print(f"   Remaining epochs: {remaining}")
        print(f"   Estimated sessions needed: {remaining // 15 + 1}")
        print("\n   To continue: Start new session and re-run Cell 7")
    else:
        print("\n✅ Training Complete!")

print("\n🔜 Next Steps:")
print("   1. Download ONNX model from Google Drive")
print("   2. Integrate into pipeline (src/detection/layout_detector.py)")
print("   3. Test inference with sample documents")
print("   4. Run end-to-end evaluation on test set")